# PPI-Cancer-Prediction Notebook

The idea of this project emerged when I found out that my newly discovered interest for networks could be combined with my rooted love for machine learning and biology to try and approach existing problems with new solutions and perspectives. 

In [103]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

from xgboost import XGBClassifier

from scipy.stats import uniform, randint


## 1. Data Exploration
### 1.1 Protein Aliases Dataset

In [3]:
df1 = pd.read_csv("C:\Local Programming Projects\ppi-cancer-prediction\data\9606.protein.aliases.v12.0.txt", sep='\t')
df1.head()

,#string_protein_id,alias,source
0,9606.ENSP00000000233,2B6H,Ensembl_PDB
1,9606.ENSP00000000233,2B6H,UniProt_DR_PDB
2,9606.ENSP00000000233,381,Ensembl_HGNC_entrez_id
3,9606.ENSP00000000233,381,KEGG_GENEID
4,9606.ENSP00000000233,381,KEGG_KEGGID_SHORT


In [4]:
print("Shape of dataset: ", df1.shape)
print("Column labels: ", df1.columns)
df1.info()
df1.describe()

Shape of dataset:  (3889207, 3)
Column labels:  Index(['#string_protein_id', 'alias', 'source'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 3889207 entries, 0 to 3889206
Data columns (total 3 columns):
 #   Column              Dtype
---  ------              -----
 0   #string_protein_id  str  
 1   alias               str  
 2   source              str  
dtypes: str(3)
memory usage: 89.0 MB


,#string_protein_id,alias,source
count,3889207,3889201,3889207
unique,19699,2553437,90
top,9606.ENSP00000354728,R-HSA-162582,Ensembl_EMBL
freq,92796,5472,487265


In [5]:
df1['#string_protein_id'].value_counts()

#string_protein_id
9606.ENSP00000354728    92796
9606.ENSP00000354982    78720
9606.ENSP00000355046    65409
9606.ENSP00000355265    54408
9606.ENSP00000354665    53281
                        ...  
9606.ENSP00000482263        9
9606.ENSP00000482855        8
9606.ENSP00000484918        8
9606.ENSP00000501013        8
9606.ENSP00000501098        8
Name: count, Length: 19699, dtype: int64

In [6]:
df1.isnull().sum().sort_values(ascending=False)

alias                 6
#string_protein_id    0
source                0
dtype: int64

### 1.2 Protein Links Dataset

In [7]:
df2 = pd.read_csv("C:\Local Programming Projects\ppi-cancer-prediction\data\9606.protein.links.v12.0.txt.gz", sep=' ')

In [8]:
print("Shape of dataset: ", df2.shape)
print("Column labels: ", df2.columns)
df2.info()
df2.describe()

Shape of dataset:  (13715404, 3)
Column labels:  Index(['protein1', 'protein2', 'combined_score'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 13715404 entries, 0 to 13715403
Data columns (total 3 columns):
 #   Column          Dtype
---  ------          -----
 0   protein1        str  
 1   protein2        str  
 2   combined_score  int64
dtypes: int64(1), str(2)
memory usage: 313.9 MB


,combined_score
count,1.371540e+07
mean,2.686240e+02
std,1.569828e+02
min,1.500000e+02
25%,1.710000e+02
50%,2.090000e+02
75%,2.980000e+02
max,9.990000e+02


In [9]:
df2.isnull().sum().sort_values(ascending=False)

protein1          0
protein2          0
combined_score    0
dtype: int64

In [10]:
df2['protein1'].value_counts().head()

protein1
9606.ENSP00000380070    9357
9606.ENSP00000494750    7773
9606.ENSP00000269305    7443
9606.ENSP00000451828    7179
9606.ENSP00000478887    7147
Name: count, dtype: int64

### 1.3 Cosmic Cancer Gene Census Dataset

In [11]:
df3 = pd.read_csv("C:\Local Programming Projects\ppi-cancer-prediction\data\Cosmic_CancerGeneCensus_v103_GRCh37.tsv", sep='\t')
df3.head(5)

,GENE_SYMBOL,NAME,COSMIC_GENE_ID,CHROMOSOME,GENOME_START,GENOME_STOP,CHR_BAND,SOMATIC,GERMLINE,TUMOUR_TYPES_SOMATIC,...,CANCER_SYNDROME,TISSUE_TYPE,MOLECULAR_GENETICS,ROLE_IN_CANCER,MUTATION_TYPES,TRANSLOCATION_PARTNER,OTHER_GERMLINE_MUT,OTHER_SYNDROME,TIER,SYNONYMS
0,A1CF,APOBEC1 complementation factor,COSG46891,10,52559169.0,52645435.0,10q11.23,y,n,melanoma,...,NaN,E,NaN,oncogene,Mis,NaN,n,NaN,2,"A1CF,ENSG00000148584.10,Q9NQ94,29974,ACF,ACF64..."
1,ABI1,abl interactor 1,COSG5120,10,27035522.0,27150016.0,10p12.1,y,n,AML,...,NaN,L,Dom,"TSG, fusion",T,KMT2A,n,NaN,1,"ABI1,ENSG00000136754.12,Q8IZP0,10006,ABI-1,E3B1"
2,ABL1,"ABL proto-oncogene 1, non-receptor tyrosine ki...",COSG4968,9,133589333.0,133763062.0,9q34.12,y,n,"CML, ALL, T-ALL",...,NaN,L,Dom,"oncogene, fusion","T, Mis","BCR, ETV6, NUP214",n,NaN,1,"ABL1,ENSG00000097007.13,P00519,25,JTK7,c-ABL,p150"
3,ABL2,"ABL proto-oncogene 2, non-receptor tyrosine ki...",COSG36573,1,179068462.0,179198819.0,1q25.2,y,n,AML,...,NaN,L,Dom,"oncogene, fusion",T,ETV6,n,NaN,1,"ABL2,ENSG00000143322.15,P42684,27,ARG"
4,ACKR3,atypical chemokine receptor 3,COSG38910,2,237476430.0,237491001.0,2q37.3,y,n,lipoma,...,NaN,M,Dom,"oncogene, fusion",T,HMGA2,n,NaN,1,"ACKR3,ENSG00000144476.5,P25106,57007,GPR159,RDC1"


In [12]:
print("Shape of dataset: ", df3.shape)
print("Column labels: ", df3.columns)
df3.info()
df3.describe()

Shape of dataset:  (763, 21)
Column labels:  Index(['GENE_SYMBOL', 'NAME', 'COSMIC_GENE_ID', 'CHROMOSOME', 'GENOME_START',
       'GENOME_STOP', 'CHR_BAND', 'SOMATIC', 'GERMLINE',
       'TUMOUR_TYPES_SOMATIC', 'TUMOUR_TYPES_GERMLINE', 'CANCER_SYNDROME',
       'TISSUE_TYPE', 'MOLECULAR_GENETICS', 'ROLE_IN_CANCER', 'MUTATION_TYPES',
       'TRANSLOCATION_PARTNER', 'OTHER_GERMLINE_MUT', 'OTHER_SYNDROME', 'TIER',
       'SYNONYMS'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 763 entries, 0 to 762
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   GENE_SYMBOL            763 non-null    str    
 1   NAME                   763 non-null    str    
 2   COSMIC_GENE_ID         763 non-null    str    
 3   CHROMOSOME             763 non-null    str    
 4   GENOME_START           757 non-null    float64
 5   GENOME_STOP            757 non-null    float64
 6   CHR_BAND               763 non-

,GENOME_START,GENOME_STOP,TIER
count,7.570000e+02,7.570000e+02,763.000000
mean,7.483567e+07,7.496238e+07,1.226737
std,5.512818e+07,5.514190e+07,0.418995
min,2.183560e+05,2.568150e+05,1.000000
25%,3.328634e+07,3.329705e+07,1.000000
50%,5.721082e+07,5.742934e+07,1.000000
75%,1.139303e+08,1.141214e+08,1.000000
max,2.436515e+08,2.440144e+08,2.000000


In [13]:
df3.isnull().sum().sort_values(ascending=False)

CANCER_SYNDROME          672
OTHER_SYNDROME           670
TUMOUR_TYPES_GERMLINE    655
TRANSLOCATION_PARTNER    392
MOLECULAR_GENETICS       170
ROLE_IN_CANCER            45
TUMOUR_TYPES_SOMATIC      44
GENOME_START               6
GENOME_STOP                6
MUTATION_TYPES             1
COSMIC_GENE_ID             0
NAME                       0
GENE_SYMBOL                0
CHROMOSOME                 0
CHR_BAND                   0
SOMATIC                    0
GERMLINE                   0
TISSUE_TYPE                0
OTHER_GERMLINE_MUT         0
TIER                       0
SYNONYMS                   0
dtype: int64

In [14]:
df3['TUMOUR_TYPES_SOMATIC'].value_counts().head()

TUMOUR_TYPES_SOMATIC
AML         39
T-ALL       19
prostate    14
ALL         12
melanoma    11
Name: count, dtype: int64

## 2. Data Preparation


In [15]:
# Drop missing values 

df1 = df1.dropna(subset=['alias'])

In [16]:
df1['source'].value_counts().head(20)

source
Ensembl_EMBL                     487265
Ensembl_protein_id               455461
Ensembl_RefSeq_short             225974
Ensembl_RefSeq                   225974
Ensembl_transcript               152417
Ensembl_HGNC_trans_name          151338
Ensembl_UCSC                     148513
Ensembl_archive_transcript       144556
Ensembl_archive_translation      125205
Ensembl_Reactome_gene            110630
UniProt_DR_RefSeq                109660
Ensembl_Reactome                 108642
Ensembl_translation               99376
Ensembl_UniProt                   96363
UniProt_AC                        88155
Ensembl_UniParc                   86072
Ensembl_PDB                       57521
UniProt_DR_PDB                    56149
KEGG_NAME_SYNONYM                 51083
Ensembl_external_synonym_HGNC     42690
Name: count, dtype: int64

In [17]:
# HGNC is the HUGO Gene Nomenclature Committee
# The Cancer Gene Census dataset only uses HGNC approved gene symbols
df_alias_gene = df1[df1['source'] == 'Ensembl_HGNC_symbol']
df_alias_gene = df_alias_gene[['#string_protein_id', 'alias']].rename(columns={'alias': 'GENE_SYMBOL'})
df_alias_gene.head()

,#string_protein_id,GENE_SYMBOL
24,9606.ENSP00000000233,ARF5
220,9606.ENSP00000000412,M6PR
398,9606.ENSP00000001008,FKBP4
528,9606.ENSP00000001146,CYP26B1
760,9606.ENSP00000002125,NDUFAF7


In [18]:
ppi = df2.copy()
ppi = ppi.merge(
    df_alias_gene,
    left_on='protein1',
    right_on='#string_protein_id',
    how='left'
).rename(columns={'GENE_SYMBOL': 'gene1'})

ppi = ppi.merge(
    df_alias_gene,
    left_on='protein2',
    right_on='#string_protein_id',
    how='left'
).rename(columns={'GENE_SYMBOL': 'gene2'})

ppi = ppi.drop(columns=['#string_protein_id_x', '#string_protein_id_y', 'protein1', 'protein2'])

In [19]:
aml_genes = df3[df3['TUMOUR_TYPES_SOMATIC'].str.contains('AML', na=False)]['GENE_SYMBOL']
aml_genes.head()

1         ABI1
3         ABL2
6        ACSL6
25    ARHGAP26
30    ARHGEF12
Name: GENE_SYMBOL, dtype: str

In [20]:
ppi_labeled = ppi.copy()

ppi_labeled['gene1_is_AML'] = ppi_labeled['gene1'].isin(aml_genes).astype(int)
ppi_labeled['gene2_is_AML'] = ppi_labeled['gene2'].isin(aml_genes).astype(int)

ppi_labeled['AML_interaction'] = ((ppi_labeled['gene1_is_AML'] == 1) |(ppi_labeled['gene2_is_AML'] == 1)).astype(int)

ppi_labeled  = ppi_labeled.reindex(columns=['gene1', 'gene2', 'combined_score', 'gene1_is_AML', 'gene2_is_AML', 'AML_interaction'])

ppi_labeled.head()

,gene1,gene2,combined_score,gene1_is_AML,gene2_is_AML,AML_interaction
0,ARF5,RALGPS2,173,0,0,0
1,ARF5,FHDC1,154,0,0,0
2,ARF5,ATP6V1E1,151,0,0,0
3,ARF5,CYTH2,471,0,0,0
4,ARF5,PSD3,201,0,0,0


## 3. Build Network

In [21]:
G = nx.from_pandas_edgelist(
    ppi_labeled,
    source = 'gene1',
    target = 'gene2',
    edge_attr='combined_score'
)

In [22]:
# Add AML labels to nodes
aml_set = set(aml_genes)

for node in G.nodes():
    G.nodes[node]['is_AML'] = int(node in aml_set)

## 4. Create Features from the Network

In [23]:
degree_dict = dict(G.degree())

weighted_degree = dict(G.degree(weight='combined_score'))

centrality = nx.degree_centrality(G)

aml_neighbors = {}

for node in G.nodes():
    neighbors = list(G.neighbors(node))
    aml_neighbors[node] = sum(
        1 for n in neighbors if G.nodes[n]['is_AML'] == 1
    )

# See how strong interactions are with AML related nodes
aml_influence = {}

for node in G.nodes():
    aml_influence[node] = sum(
        G[node][n]['combined_score']
        for n in G.neighbors(node)
        if G.nodes[n]['is_AML'] == 1
    )

In [24]:
total_nodes = len(G.nodes())
total_aml = sum(G.nodes[n]['is_AML'] for n in G.nodes())

aml_ratio = total_aml / total_nodes

network_data = pd.DataFrame({
    'gene': list(G.nodes()),
    'centrality': [centrality[n] for n in G.nodes()],
    'aml_neighbor_fraction': [
        aml_neighbors[n] / max(degree_dict[n], 1)
        for n in G.nodes()
    ],

    'aml_influence_norm': [
        aml_influence[n] / max(weighted_degree[n], 1)
        for n in G.nodes()
    ],

    'aml_neighbor_residual': [
        aml_neighbors[n] - degree_dict[n] * aml_ratio
        for n in G.nodes()
    ],

    # --- Target ---
    'is_AML': [G.nodes[n]['is_AML'] for n in G.nodes()]
})

network_data.head()

,gene,centrality,aml_neighbor_fraction,aml_influence_norm,aml_neighbor_residual,is_AML
0,ARF5,0.067014,0.003891,0.003803,-0.561900,0
1,RALGPS2,0.035828,0.007278,0.009761,2.026439,0
2,FHDC1,0.037914,0.013755,0.013790,6.853306,0
3,ATP6V1E1,0.050535,0.002064,0.002187,-2.194149,0
4,CYTH2,0.016532,0.006309,0.004761,0.627920,0


## 5. Model Selection

In [25]:
X = network_data.drop(columns=['gene', 'is_AML'])
y = network_data['is_AML']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Scale features such that logistic regression model doesn't become biased to large ranges
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # don't fit test data 


In [73]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

### 5.1 Logistic Regression Model

In [ ]:
LR_model = LogisticRegression(max_iter=1000)

LR_scores = cross_val_score(LR_model, X, y, cv=cv, scoring='roc_auc')

print(LR_scores.mean())

LR_model.fit(X_train_scaled, y_train)

0.9634509029179403


In [76]:
LR_model.fit(X_train_scaled, y_train)

y_pred = LR_model.predict(X_test_scaled)

y_prob = LR_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
print(classification_report(y_test, y_pred))

roc_auc_lr = roc_auc_score(y_test, y_prob)
print("ROC-AUC:", roc_auc_lr)

pr_auc_lr = average_precision_score(y_test, y_prob)
print(f"PR-AUC for LR: {pr_auc_lr}")

              precision    recall  f1-score   support

           0       1.00      0.99      0.99      3819
           1       0.22      0.65      0.33        17

    accuracy                           0.99      3836
   macro avg       0.61      0.82      0.66      3836
weighted avg       0.99      0.99      0.99      3836

ROC-AUC: 0.9691788734346841
PR-AUC: 0.3028008934302157


In [ ]:
# X_scaled = scaler.transform(X)

# network_data['aml_probability'] = LR_model.predict_proba(X_scaled)[:, 1]

# # Sort genes by likelihood
# ranked_genes = network_data.sort_values(by='aml_probability', ascending=False)

# ranked_genes.head(10)

,gene,centrality,aml_neighbor_fraction,aml_influence_norm,aml_neighbor_residual,is_AML,aml_probability
10761,MLLT10,0.030352,0.097938,0.143486,54.480914,0,0.984567
9024,ETV6,0.052516,0.061569,0.105388,57.641375,0,0.964788
10795,MLLT6,0.018774,0.122222,0.147011,42.441802,0,0.924520
10836,HOXA9,0.050639,0.061792,0.089470,55.797194,1,0.910565
10782,MLLT3,0.056271,0.057461,0.073528,57.329735,0,0.855908
10747,AFF1,0.050065,0.058333,0.081065,51.844806,0,0.822072
14109,SETBP1,0.053090,0.047151,0.094562,43.593763,1,0.813471
10855,MLLT11,0.013559,0.123077,0.150261,30.874635,0,0.794077
10853,MECOM,0.049909,0.055381,0.081423,48.857791,1,0.780551
3367,RUNX1,0.114941,0.030399,0.053561,57.460367,1,0.764890


In [ ]:
# threshold = 0.1  

# y_pred_custom = (y_prob > threshold).astype(int)

# print(classification_report(y_test, y_pred_custom))

# roc_auc = roc_auc_score(y_test, y_pred_custom)
# print("ROC-AUC:", roc_auc)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3819
           1       0.46      0.65      0.54        17

    accuracy                           1.00      3836
   macro avg       0.73      0.82      0.77      3836
weighted avg       1.00      1.00      1.00      3836

ROC-AUC: 0.8218273955300894


## 5.2 Random Forest Model 

In [98]:
RF_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_scores = cross_val_score(RF_model, X, y, cv=cv, scoring='roc_auc')

print(rf_scores)
print(rf_scores.mean())

[0.94232398 0.97894573 0.94438171 0.93153175 0.92079315]
0.9435952652320768


In [ ]:
RF_model.fit(X_train, y_train)
y_pred = RF_model.predict(X_test)
y_prob = RF_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
roc_auc_rf = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC: {roc_auc_rf}")

pr_auc_rf = average_precision_score(y_test, y_prob)
print(f"PR-AUC for RF: {pr_auc_rf}")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3819
           1       0.29      0.41      0.34        17

    accuracy                           0.99      3836
   macro avg       0.64      0.70      0.67      3836
weighted avg       0.99      0.99      0.99      3836

ROC-AUC: 0.9428322782372964
PR-AUC: 0.26865355277419506


## 5.3 XGBoost Model 

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    objective='binary:logistic',
    scale_pos_weight=len(y[y==0]) / len(y[y==1]),  
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

xgb_scores = cross_val_score(xgb_model, X, y, cv=cv, scoring='roc_auc')
print(xgb_scores.mean())

0.9560011661213013


In [ ]:
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
roc_auc_xgb = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC: {roc_auc_xgb}")

pr_auc_xgb = average_precision_score(y_test, y_prob)
print(f"PR-AUC for XGB: {pr_auc_xgb}")

In [104]:
param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(3, 8),
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.6, 0.4),        # samples between 0.6-1.0
    'colsample_bytree': uniform(0.6, 0.4), # features between 0.6-1.0
    'min_child_weight': randint(1, 10),
    'gamma': uniform(0, 0.5)
}

xgb_base = XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=len(y[y==0]) / len(y[y==1]),
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=50,           # tries 50 random combinations
    scoring='average_precision',  # optimizes for PR-AUC directly
    cv=cv,               # your existing StratifiedKFold
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Best CV PR-AUC:", random_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'colsample_bytree': np.float64(0.9376852562905246), 'gamma': np.float64(0.46500841740541593), 'learning_rate': np.float64(0.02408322616990878), 'max_depth': 4, 'min_child_weight': 6, 'n_estimators': 250, 'subsample': np.float64(0.6560336060946096)}
Best CV PR-AUC: 0.28484215094427057


In [105]:
best_xgb = random_search.best_estimator_

y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       1.00      0.97      0.99      3819
           1       0.11      0.71      0.19        17

    accuracy                           0.97      3836
   macro avg       0.55      0.84      0.59      3836
weighted avg       0.99      0.97      0.98      3836

ROC-AUC: 0.9732914375490966
PR-AUC: 0.3572573058057555


# NOTE

- XGB is best, with slightly lower precision (meaning more false-positives, which is okay in this scenario), but higher recall meaning it predicts correctly more real AML genes
- ROC-AUC is misleading due to imbalanced dataset (many non-AML genes, therefore, model easily predicts those correctly)
- PR-AUC is more useful for this dataset